## Exercise 1

In this exercise, we will predict the number of applications received using the other variables in the `College` data set:

- (a) Split the data set into a training set and a test set.

- (b) Fit a linear model using least squares on the training set, and report the test error obtained.

- (c) Fit a ridge regression model on the training set, with `lambda` chosen by cross-validation. Report the test error obtained.

- (d) Fit a lasso model on the training set, with `lambda` chosen by crossvalidation. Report the test error obtained, along with the number of non-zero coefficient estimates.

- (e) Comment on the results obtained. How accurately can we predict the number of college applications received? Is there much difference among the test errors resulting from these three approaches?

In [2]:
# Required packages

import numpy as np
import pandas as pd
import statsmodels.api as sm
import sklearn.model_selection as skm
import sklearn.linear_model as skl
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from ISLP import load_data
from ISLP.models import ModelSpec as MS
from ISLP.models import sklearn_sm

In [3]:
# Import from ISLP

college = load_data('College')
print(college.shape)
college.describe()

(777, 18)


,Apps,Accept,Enroll,Top10perc,Top25perc,F.Undergrad,P.Undergrad,Outstate,Room.Board,Books,Personal,PhD,Terminal,S.F.Ratio,perc.alumni,Expend,Grad.Rate
count,777.000000,777.000000,777.000000,777.000000,777.000000,777.000000,777.000000,777.000000,777.000000,777.000000,777.000000,777.000000,777.000000,777.000000,777.000000,777.000000,777.00000
mean,3001.638353,2018.804376,779.972973,27.558559,55.796654,3699.907336,855.298584,10440.669241,4357.526384,549.380952,1340.642214,72.660232,79.702703,14.089704,22.743887,9660.171171,65.46332
std,3870.201484,2451.113971,929.176190,17.640364,19.804778,4850.420531,1522.431887,4023.016484,1096.696416,165.105360,677.071454,16.328155,14.722359,3.958349,12.391801,5221.768440,17.17771
min,81.000000,72.000000,35.000000,1.000000,9.000000,139.000000,1.000000,2340.000000,1780.000000,96.000000,250.000000,8.000000,24.000000,2.500000,0.000000,3186.000000,10.00000
25%,776.000000,604.000000,242.000000,15.000000,41.000000,992.000000,95.000000,7320.000000,3597.000000,470.000000,850.000000,62.000000,71.000000,11.500000,13.000000,6751.000000,53.00000
50%,1558.000000,1110.000000,434.000000,23.000000,54.000000,1707.000000,353.000000,9990.000000,4200.000000,500.000000,1200.000000,75.000000,82.000000,13.600000,21.000000,8377.000000,65.00000
75%,3624.000000,2424.000000,902.000000,35.000000,69.000000,4005.000000,967.000000,12925.000000,5050.000000,600.000000,1700.000000,85.000000,92.000000,16.500000,31.000000,10830.000000,78.00000
max,48094.000000,26330.000000,6392.000000,96.000000,100.000000,31643.000000,21836.000000,21700.000000,8124.000000,2340.000000,6800.000000,103.000000,100.000000,39.800000,64.000000,56233.000000,118.00000


### (a) Training set / test set


In [20]:
# Design matrix and train/test split

outer_valid = skm.ShuffleSplit(n_splits=1,
                               test_size=0.5,
                               random_state=1)

design = MS(college.columns.drop('Apps'))
D = design.fit_transform(college)
D = D.drop('intercept', axis=1)
X = np.asarray(D)
Y = np.array(college['Apps'])

def report(name, results):
    """Print the test RMSE of a model evaluated on outer_valid and return it."""
    rmse = np.sqrt(-results['test_score'][0])
    print(f"Test RMSE of the {name}: {rmse:.2f}")
    print(f"This is {rmse / rmse_null * 100:.2f} % of the RMSE of the null model.")
    return rmse

In [21]:
# Benchmark: null model that always predicts the training mean

results = skm.cross_validate(DummyRegressor(),
                             X,
                             Y,
                             cv=outer_valid,
                             scoring='neg_mean_squared_error')
rmse_null = np.sqrt(-results['test_score'][0])
print(f"Test RMSE of the null model: {rmse_null:.2f}")

Test RMSE of the null model: 4178.00


### (b) Least squares

In [22]:
# OLS model (no scaling needed: it does not change the least-squares fit)

ols = sklearn_sm(sm.OLS)

results = skm.cross_validate(ols,
                             X,
                             Y,
                             cv=outer_valid,
                             scoring='neg_mean_squared_error')
rmse_ols = report("OLS model", results)

Test RMSE of the OLS model: 1192.99
This is 28.55 % of the RMSE of the null model.


### (c) Ridge regression

In [16]:
# Define the lambda grid and look at the coefficient path

lambdas = 10**np.linspace(6, -4, 100) 

Xs = (X - X.mean(0)) / X.std(0)

lambdas, soln_array = skl.ElasticNet.path(Xs, 
                                    Y,
                                    l1_ratio=0,
                                    alphas=lambdas)[:2]
soln_path = pd.DataFrame(soln_array.T,
                         columns=D.columns,
                         index=-np.log(lambdas))
soln_path

,Private[Yes],Accept,Enroll,Top10perc,Top25perc,F.Undergrad,P.Undergrad,Outstate,Room.Board,Books,Personal,PhD,Terminal,S.F.Ratio,perc.alumni,Expend,Grad.Rate
-13.815511,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
-13.582926,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
-13.350342,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
-13.117757,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
-12.885173,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8.280003,-220.366169,3871.114278,-797.894718,875.509320,-278.354400,270.694232,67.490009,-343.320551,166.308222,3.560088,20.800457,-141.068219,-49.342449,60.929593,1.007913,406.826699,149.160281
8.512587,-220.366169,3871.114278,-797.894718,875.509320,-278.354400,270.694232,67.490009,-343.320551,166.308222,3.560088,20.800457,-141.068219,-49.342449,60.929593,1.007913,406.826699,149.160281
8.745172,-220.239462,3876.582933,-805.667450,877.422543,-279.760307,273.405304,67.558630,-344.200800,166.009209,3.510382,20.897866,-141.298216,-49.193735,60.920051,1.502416,406.739414,149.026826
8.977756,-220.239462,3876.582933,-805.667450,877.422543,-279.760307,273.405304,67.558630,-344.200800,166.009209,3.510382,20.897866,-141.298216,-49.193735,60.920051,1.502416,406.739414,149.026826


In [23]:
# Ridge: lambda chosen by 5-fold CV on the training set

inner_cv = skm.KFold(n_splits=5,
                     shuffle=True,
                     random_state=6897)

ridgeCV = skl.RidgeCV(alphas=lambdas,
                     cv=inner_cv)

pipeCV = Pipeline(steps=[('scaler', StandardScaler()),
                         ('ridge', ridgeCV)])

results = skm.cross_validate(pipeCV,
                             X,
                             Y,
                             cv=outer_valid,
                             scoring='neg_mean_squared_error',
                             return_estimator=True)
rmse_ridge = report("ridge model", results)

# Check that the chosen lambda is not on the edge of the grid
ridge_fit = results['estimator'][0].named_steps['ridge']
print(f"\nChosen lambda: {ridge_fit.alpha_:.4g}")
print(f"Grid range: [{lambdas.min():.4g}, {lambdas.max():.4g}]")

Test RMSE of the ridge model: 1248.77
This is 29.89 % of the RMSE of the null model.

Chosen lambda: 5.591
Grid range: [0.0001, 1e+06]


### (d) Lasso

In [24]:
# Lasso: lambda chosen by 5-fold CV on the training set

lassoCV = skl.ElasticNetCV(alphas=100,
                           l1_ratio=1,
                           cv=inner_cv,
                           max_iter=50000)

pipeCV = Pipeline(steps=[('scaler', StandardScaler()),
                         ('lasso', lassoCV)])

results = skm.cross_validate(pipeCV,
                             X,
                             Y,
                             cv=outer_valid,
                             scoring='neg_mean_squared_error',
                             return_estimator=True)
rmse_lasso = report("lasso model", results)

lasso_fit = results['estimator'][0].named_steps['lasso']
print(f"\nChosen lambda: {lasso_fit.alpha_:.4g}")
print(f"Grid range: [{lasso_fit.alphas_.min():.4g}, {lasso_fit.alphas_.max():.4g}]")

n_nonzero = np.sum(lasso_fit.coef_ != 0)
print(f"Non-zero coefficients: {n_nonzero} out of {X.shape[1]}")
pd.Series(lasso_fit.coef_, index=D.columns).round(2)

Test RMSE of the lasso model: 1187.85
This is 28.43 % of the RMSE of the null model.

Chosen lambda: 7.472
Grid range: [3.234, 3234]
Non-zero coefficients: 16 out of 17


Private[Yes]    -195.78
Accept          2541.16
Enroll           416.24
Top10perc       1071.42
Top25perc       -382.63
F.Undergrad        0.00
P.Undergrad        9.77
Outstate        -240.31
Room.Board       259.14
Books            -23.82
Personal         -47.52
PhD              -82.41
Terminal         -54.35
S.F.Ratio         15.09
perc.alumni     -118.36
Expend           325.74
Grad.Rate        187.52
dtype: float64

### (e) Comparison

The three models (OLS, ridge, LASSO) divide prediction errors by approximately 3 compared to the null model always predicting the mean. The improvement in prediction accuracy of the ridge and LASSO models is very small relative to the OLS model. This illustrates that regularization methods are mostly useful in settings with a large number of predictors (high variance) which is not the case here.